In [1]:
"""
============================================================
UCI HAR (Human Activity Recognition) Classification
LW-AdaDelta vs Baselines — Variable-Length Temporal Benchmark
============================================================

Dataset : UCI HAR (Human Activity Recognition Using Smartphones)
          10,299 samples, 561 features (time + frequency domain)
          6 classes: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS,
                     SITTING, STANDING, LAYING
Source  : https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones

Task    : 6-class activity classification from inertial sensor windows
          (accelerometer + gyroscope, 50Hz, 2.56s windows = 128 timesteps)

Why UCI HAR for LW-AdaDelta?
  ✓ Real sensor data with natural measurement noise
  ✓ Variable difficulty: static activities (SITTING/STANDING) vs
    dynamic transitions (WALKING_UP/DOWN) stress temporal features
  ✓ Multivariate time series — 9 raw signals × 128 timesteps
  ✓ Feature match: Variable_Length + Long_Dependency wins in synthetic bench

Features:
✓ Raw inertial signals used (not pre-extracted 561 features)
  → forces the model to learn temporal structure, stressing the optimizer
✓ Early stopping on validation accuracy
✓ Multiple seeds for statistical validity
✓ Metrics: Accuracy, Macro F1, per-class F1 breakdown
✓ Fair comparison across 6 optimizers

Setup:
  pip install torch numpy pandas scipy scikit-learn matplotlib requests
"""

import os
import math
import time
import zipfile
import warnings
from collections import deque
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ============================================================
# LW-AdaDelta Optimizer (identical across all benchmarks)
# ============================================================

class LWAdaDelta(torch.optim.Optimizer):
    """Local-Window AdaDelta with temporal smoothing"""

    def __init__(self, params, rho=0.9, eps=1e-6, k=2, tau=1.0):
        defaults = dict(rho=rho, eps=eps, k=k, tau=tau)
        super().__init__(params, defaults)
        self._weight_cache = {}

    def _weights(self, k, tau, device):
        key = (k, tau, device)
        if key not in self._weight_cache:
            w = torch.tensor(
                [math.exp(-i / tau) for i in range(k)],
                device=device
            )
            self._weight_cache[key] = w / w.sum()
        return self._weight_cache[key]

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            rho, eps, k, tau = group["rho"], group["eps"], group["k"], group["tau"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad  = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["Eg2"]  = torch.zeros_like(p)
                    state["Edx2"] = torch.zeros_like(p)
                    state["buf"]  = deque(maxlen=k)

                Eg2, Edx2, buf = state["Eg2"], state["Edx2"], state["buf"]
                Eg2.mul_(rho).addcmul_(grad, grad, value=1 - rho)

                rms_dx    = torch.sqrt(Edx2 + eps)
                rms_g     = torch.sqrt(Eg2  + eps)
                delta_raw = -(rms_dx / rms_g) * grad
                buf.append(delta_raw.detach())

                weights = self._weights(len(buf), tau, p.device)
                delta   = torch.zeros_like(p)
                for w, u in zip(weights, reversed(buf)):
                    delta.add_(u, alpha=w.item())

                p.add_(delta)
                Edx2.mul_(rho).addcmul_(delta, delta, value=1 - rho)


class Lion(torch.optim.Optimizer):
    """Lion optimizer"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad    = p.grad
                state   = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg      = state['exp_avg']
                beta1, beta2 = group['betas']
                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


# ============================================================
# Data Download & Loading
# ============================================================

UCI_HAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)

ACTIVITY_LABELS = {
    1: 'WALKING',
    2: 'WALKING_UPSTAIRS',
    3: 'WALKING_DOWNSTAIRS',
    4: 'SITTING',
    5: 'STANDING',
    6: 'LAYING'
}

# Raw signal files (9 channels: body_acc xyz, body_gyro xyz, total_acc xyz)
RAW_SIGNAL_FILES = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]


def download_uci_har(data_dir='./uci_har_data'):
    """Download and extract UCI HAR dataset."""
    import urllib.request

    zip_path     = os.path.join(data_dir, 'uci_har.zip')
    extract_path = os.path.join(data_dir, 'UCI HAR Dataset')

    if os.path.exists(extract_path):
        print(f"[INFO] UCI HAR already extracted at: {extract_path}")
        return extract_path

    os.makedirs(data_dir, exist_ok=True)

    if not os.path.exists(zip_path):
        print("[INFO] Downloading UCI HAR Dataset (~60 MB) …")
        try:
            urllib.request.urlretrieve(UCI_HAR_URL, zip_path)
            print(f"[INFO] Downloaded → {zip_path}")
        except Exception as e:
            raise RuntimeError(
                f"Download failed: {e}\n\n"
                "Manual download:\n"
                "  1. https://archive.ics.uci.edu/ml/datasets/"
                "human+activity+recognition+using+smartphones\n"
                "  2. Extract ZIP to './uci_har_data/UCI HAR Dataset/'\n"
            )

    print("[INFO] Extracting …")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)
    print(f"[INFO] Extracted → {extract_path}")
    return extract_path


def load_raw_signals(data_dir, split='train'):
    """
    Load the 9-channel raw inertial signals for a split.

    Each file: [N_samples, 128]  (128 timesteps at 50Hz = 2.56s window)
    Stack to:  [N_samples, 9, 128]

    We use raw signals (not the pre-computed 561 features) to force the
    model to learn temporal structure — this is the setting that most
    stresses the optimizer's handling of temporal dependencies.
    """
    split_dir    = os.path.join(data_dir, split)
    signals_dir  = os.path.join(split_dir, 'Inertial Signals')

    channels = []
    for sig_name in RAW_SIGNAL_FILES:
        fname = os.path.join(signals_dir, f'{sig_name}_{split}.txt')
        arr   = np.loadtxt(fname, dtype=np.float32)   # [N, 128]
        channels.append(arr)

    X = np.stack(channels, axis=1)   # [N, 9, 128]

    # Labels (1-indexed → 0-indexed)
    label_file = os.path.join(split_dir, 'y_' + split + '.txt')
    y = np.loadtxt(label_file, dtype=np.int64) - 1   # [N]  → 0..5

    print(f"[INFO] {split}: X={X.shape}, y={y.shape}, "
          f"classes={np.unique(y).tolist()}")
    return X, y


def load_uci_har(data_dir):
    """
    Load train and test splits, normalize per-channel.

    Normalization: per-channel z-score on train statistics only.
    Test set is kept clean (original distribution).

    Returns:
        X_train, y_train : [7352, 9, 128], [7352]
        X_test,  y_test  : [2947, 9, 128], [2947]
    """
    X_train, y_train = load_raw_signals(data_dir, split='train')
    X_test,  y_test  = load_raw_signals(data_dir, split='test')

    # Per-channel normalization (fit on train, apply to test)
    # Shape: [1, 9, 1] for broadcasting
    mean = X_train.mean(axis=(0, 2), keepdims=True)   # [1, 9, 1]
    std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8

    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    print(f"\n[INFO] After normalization:")
    print(f"       Train: {X_train.shape}  Test: {X_test.shape}")
    print(f"       Class distribution (train): "
          + str({ACTIVITY_LABELS[i+1]: int((y_train==i).sum())
                 for i in range(6)}))

    return X_train, y_train, X_test, y_test


# ============================================================
# Dataset & DataLoader
# ============================================================

class HARDataset(Dataset):
    """
    UCI HAR dataset.
    Each sample: x=[9, 128] (9 channels × 128 timesteps), y=int (0–5)
    """
    def __init__(self, X, y, augment=False):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.augment:
            # Mild Gaussian noise on sensor readings
            x = x + torch.randn_like(x) * 0.02
            # Random time shift (roll along timestep axis)
            shift = torch.randint(-8, 8, (1,)).item()
            x = torch.roll(x, shift, dims=1)
        return x, y


def make_loaders(X_train, y_train, X_test, y_test,
                 seed=42, val_ratio=0.15, batch_size=128):
    """
    Split train → train/val (stratified), keep test separate.
    UCI HAR already has a fixed test split — we honor it.
    """
    from sklearn.model_selection import train_test_split

    idx = np.arange(len(y_train))
    idx_tr, idx_val = train_test_split(
        idx, test_size=val_ratio, stratify=y_train, random_state=seed
    )

    train_ds = HARDataset(X_train[idx_tr],  y_train[idx_tr],  augment=True)
    val_ds   = HARDataset(X_train[idx_val], y_train[idx_val], augment=False)
    test_ds  = HARDataset(X_test,           y_test,           augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=4, pin_memory=True)

    print(f"[INFO] Loaders — Train: {len(train_ds)}, "
          f"Val: {len(val_ds)}, Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader


# ============================================================
# Model: Multi-Scale Temporal CNN
# ============================================================
# Architecture rationale:
#
#   UCI HAR has two fundamentally different activity types:
#     • DYNAMIC  (WALKING, WALKING_UP, WALKING_DOWN): high-frequency
#       periodic motion at 1–3 Hz — captured by short kernels
#     • STATIC   (SITTING, STANDING, LAYING): slow drift and postural
#       signals — captured by longer kernels
#
#   A multi-scale CNN with parallel branches (kernel=3, 7, 15) captures
#   both simultaneously. This forces the optimizer to learn useful
#   gradient updates at multiple temporal scales — precisely the
#   condition that exposes AdaDelta's stale-history problem and where
#   LW-AdaDelta's local window provides the most benefit.
#
#   The branches are fused and passed through a temporal attention
#   pooling layer instead of simple global average pooling, so the
#   model can weight informative timesteps more heavily.

class TemporalBlock(nn.Module):
    """Single-scale temporal conv block with BN + GELU"""
    def __init__(self, in_ch, out_ch, kernel, dropout=0.2):
        super().__init__()
        pad  = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        )
        self.downsample = (
            nn.Conv1d(in_ch, out_ch, 1)
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        return self.net(x) + self.downsample(x)


class TemporalAttentionPool(nn.Module):
    """
    Attention-weighted temporal pooling.
    Learns which timesteps matter most for classification.
    Input:  [B, C, T]
    Output: [B, C]
    """
    def __init__(self, in_ch):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_ch, in_ch // 4, 1),
            nn.Tanh(),
            nn.Conv1d(in_ch // 4, 1, 1),   # [B, 1, T]
        )

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=-1)   # [B, 1, T]
        return (x * weights).sum(dim=-1)                 # [B, C]


class MultiScaleTCN(nn.Module):
    """
    Multi-scale Temporal CNN for HAR.
    Input:  [B, 9, 128]   (9 sensor channels × 128 timesteps)
    Output: [B, 6]        (6 activity classes)
    """

    def __init__(self, in_ch=9, num_classes=6, base_ch=64, dropout=0.3):
        super().__init__()

        # Three parallel branches: short, medium, long temporal scale
        self.branch_short  = TemporalBlock(in_ch, base_ch,      kernel=3,  dropout=dropout)
        self.branch_medium = TemporalBlock(in_ch, base_ch,      kernel=7,  dropout=dropout)
        self.branch_long   = TemporalBlock(in_ch, base_ch,      kernel=15, dropout=dropout)

        fused_ch = base_ch * 3   # 192

        # Fusion + deeper processing
        self.fusion = nn.Sequential(
            TemporalBlock(fused_ch, base_ch * 2, kernel=7, dropout=dropout),
            TemporalBlock(base_ch * 2, base_ch * 4, kernel=5, dropout=dropout),
        )
        # [B, 256, 128]

        self.pool = TemporalAttentionPool(base_ch * 4)   # [B, 256]

        self.classifier = nn.Sequential(
            nn.Linear(base_ch * 4, base_ch * 2),
            nn.LayerNorm(base_ch * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(base_ch * 2, num_classes),
        )

    def forward(self, x):
        # x: [B, 9, 128]
        s = self.branch_short(x)    # [B, 64, 128]
        m = self.branch_medium(x)   # [B, 64, 128]
        l = self.branch_long(x)     # [B, 64, 128]

        fused = torch.cat([s, m, l], dim=1)   # [B, 192, 128]
        fused = self.fusion(fused)            # [B, 256, 128]
        pooled = self.pool(fused)             # [B, 256]
        return self.classifier(pooled)        # [B, 6]


# ============================================================
# Optimizer Factory
# ============================================================

def get_optimizer(name, params):
    if name == 'SGD':
        return torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
    elif name == 'Adam':
        return torch.optim.Adam(params, lr=0.001, weight_decay=1e-4)
    elif name == 'AdamW':
        return torch.optim.AdamW(params, lr=0.001, weight_decay=1e-3)
    elif name == 'Lion':
        return Lion(params, lr=1e-4, weight_decay=1e-3)
    elif name == 'AdaDelta':
        return torch.optim.Adadelta(params, rho=0.9)
    elif name == 'LW-AdaDelta':
        return LWAdaDelta(params, rho=0.9, k=2, tau=1.0)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


# ============================================================
# Training Utilities
# ============================================================

def train_epoch(model, optimizer, loader, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = F.cross_entropy(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += logits.argmax(1).eq(y_b).sum().item()
        total      += y_b.size(0)

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device, return_preds=False):
    """Returns loss, accuracy, macro-F1. Optionally returns raw predictions."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss   = F.cross_entropy(logits, y_b)
        total_loss += loss.item()
        preds = logits.argmax(1)
        correct += preds.eq(y_b).sum().item()
        total   += y_b.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_b.cpu().numpy())

    acc    = 100.0 * correct / total
    macro_f1 = f1_score(all_targets, all_preds, average='macro') * 100
    per_class_f1 = f1_score(
        all_targets, all_preds, average=None, labels=list(range(6))
    ) * 100

    if return_preds:
        return (total_loss / len(loader), acc, macro_f1,
                per_class_f1, np.array(all_preds), np.array(all_targets))
    return total_loss / len(loader), acc, macro_f1, per_class_f1


def train_with_early_stopping(
    model, optimizer, train_loader, val_loader, device,
    min_epochs=30, max_epochs=200, patience=15
):
    """Train with early stopping on validation macro-F1."""
    best_val_f1  = 0.0
    best_state   = None
    no_improve   = 0

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   [],
        'val_f1':     [],
        'epoch_times': []
    }

    for epoch in range(max_epochs):
        t0 = time.time()

        tr_loss, tr_acc                     = train_epoch(model, optimizer, train_loader, device)
        val_loss, val_acc, val_f1, _        = evaluate(model, val_loader, device)
        elapsed                             = time.time() - t0

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['epoch_times'].append(elapsed)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}: "
                  f"TrainAcc={tr_acc:.2f}%  "
                  f"ValAcc={val_acc:.2f}%  "
                  f"ValF1={val_f1:.2f}%  "
                  f"Time={elapsed:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = deepcopy(model.state_dict())
            no_improve  = 0
        else:
            no_improve += 1

        if epoch >= min_epochs and no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1} "
                  f"(no F1 improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history['best_val_f1']  = best_val_f1
    history['total_epochs'] = epoch + 1
    return history


# ============================================================
# Single Experiment
# ============================================================

def run_experiment(X_train, y_train, X_test, y_test,
                   optimizer_name, seed, device):
    print(f"\n{'='*60}")
    print(f"Optimizer={optimizer_name}  Seed={seed}")
    print(f"{'='*60}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    train_loader, val_loader, test_loader = make_loaders(
        X_train, y_train, X_test, y_test,
        seed=seed, batch_size=128
    )

    model     = MultiScaleTCN(in_ch=9, num_classes=6,
                               base_ch=64, dropout=0.3).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    optimizer = get_optimizer(optimizer_name, model.parameters())

    history = train_with_early_stopping(
        model, optimizer, train_loader, val_loader, device,
        min_epochs=30, max_epochs=200, patience=15
    )

    (test_loss, test_acc, test_f1,
     per_class_f1, test_preds, test_targets) = evaluate(
        model, test_loader, device, return_preds=True
    )

    print(f"\n  Final Test Results:")
    print(f"    Accuracy  = {test_acc:.2f}%")
    print(f"    Macro F1  = {test_f1:.2f}%")
    print(f"    Per-class F1:")
    for i, f1 in enumerate(per_class_f1):
        print(f"      {ACTIVITY_LABELS[i+1]:<22}: {f1:.2f}%")
    print(f"    Epochs    = {history['total_epochs']}")

    return {
        'history':       history,
        'test_acc':      test_acc,
        'test_f1':       test_f1,
        'per_class_f1':  per_class_f1,
        'test_preds':    test_preds,
        'test_targets':  test_targets,
        'total_time':    sum(history['epoch_times'])
    }


# ============================================================
# Full Benchmark
# ============================================================

def run_full_benchmark(X_train, y_train, X_test, y_test,
                       device='cuda', seeds=[0, 1, 2]):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    results    = {opt: [] for opt in optimizers}

    total = len(optimizers) * len(seeds)
    done  = 0

    for opt_name in optimizers:
        for seed in seeds:
            done += 1
            print(f"\n[Progress {done}/{total}] {opt_name}  seed={seed}")
            result = run_experiment(
                X_train, y_train, X_test, y_test,
                optimizer_name=opt_name,
                seed=seed,
                device=device
            )
            results[opt_name].append(result)

    return results


# ============================================================
# Analysis
# ============================================================

def analyze_results(results):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']

    print("\n" + "="*80)
    print("UCI HAR — COMPREHENSIVE RESULTS ANALYSIS")
    print("Primary metric: Macro F1 (balances all 6 activity classes)")
    print("="*80)

    print(f"\n{'Optimizer':<15} {'Test Acc':<22} {'Macro F1':<22} "
          f"{'Epochs':<10} {'Time (s)'}")
    print("-"*75)

    all_metrics = {}
    for opt in optimizers:
        runs  = results[opt]
        accs  = [r['test_acc'] for r in runs]
        f1s   = [r['test_f1']  for r in runs]
        eps   = [r['history']['total_epochs'] for r in runs]
        times = [r['total_time'] for r in runs]

        # Per-class F1 averages across seeds
        per_cls = np.mean(
            np.stack([r['per_class_f1'] for r in runs], axis=0),
            axis=0
        )

        m = {
            'acc_mean':  np.mean(accs),  'acc_std':  np.std(accs),
            'f1_mean':   np.mean(f1s),   'f1_std':   np.std(f1s),
            'ep_mean':   np.mean(eps),
            'time_mean': np.mean(times),
            'per_cls_f1': per_cls,
        }
        all_metrics[opt] = m

        print(f"{opt:<15} "
              f"{m['acc_mean']:>6.2f}±{m['acc_std']:.2f}%    "
              f"{m['f1_mean']:>6.2f}±{m['f1_std']:.2f}%    "
              f"{m['ep_mean']:>5.0f}    "
              f"{m['time_mean']:>8.0f}")

    # ── Head-to-head ──────────────────────────────────────────────
    lw  = all_metrics['LW-AdaDelta']
    ada = all_metrics['AdaDelta']

    d_f1  = lw['f1_mean']  - ada['f1_mean']
    d_acc = lw['acc_mean'] - ada['acc_mean']

    print(f"\n{'='*80}")
    print("LW-ADADELTA vs ADADELTA — HEAD-TO-HEAD")
    print(f"{'='*80}")
    print(f"  Macro F1  : AdaDelta={ada['f1_mean']:.2f}%  "
          f"LW-AdaDelta={lw['f1_mean']:.2f}%  "
          f"Δ={d_f1:+.2f}%  "
          + ("✓ LW wins" if d_f1 > 0 else "✗ AdaDelta wins"))
    print(f"  Accuracy  : AdaDelta={ada['acc_mean']:.2f}%  "
          f"LW-AdaDelta={lw['acc_mean']:.2f}%  "
          f"Δ={d_acc:+.2f}%  "
          + ("✓ LW wins" if d_acc > 0 else "✗ AdaDelta wins"))

    # Per-class breakdown
    print(f"\n  Per-class F1 breakdown:")
    print(f"  {'Activity':<25} {'AdaDelta':>10} {'LW-AdaDelta':>13} {'Δ':>8}")
    print(f"  {'-'*58}")
    for i in range(6):
        name = ACTIVITY_LABELS[i+1]
        a_f1 = ada['per_cls_f1'][i]
        lw_f1 = lw['per_cls_f1'][i]
        d = lw_f1 - a_f1
        marker = " ✓" if d > 0 else ""
        print(f"  {name:<25} {a_f1:>9.2f}%  {lw_f1:>11.2f}%  "
              f"{d:>+7.2f}%{marker}")

    # Paired t-test
    lw_f1s  = [r['test_f1'] for r in results['LW-AdaDelta']]
    ada_f1s = [r['test_f1'] for r in results['AdaDelta']]
    if len(lw_f1s) >= 2:
        t, p = stats.ttest_rel(lw_f1s, ada_f1s)
        sig  = "✓ p<0.05" if p < 0.05 else "✗ not significant"
        print(f"\n  Paired t-test: t={t:.3f}  p={p:.4f}  {sig}")

    # ── Competitive check ─────────────────────────────────────────
    print(f"\n{'='*80}")
    print("COMPETITIVE PERFORMANCE (within 2% F1 of best)")
    print(f"{'='*80}")
    best_f1 = max(m['f1_mean'] for m in all_metrics.values())
    for opt, m in all_metrics.items():
        gap  = best_f1 - m['f1_mean']
        comp = "✓ competitive" if gap <= 2.0 else "✗"
        print(f"  {opt:<15} F1={m['f1_mean']:.2f}%  gap={gap:.2f}%  {comp}")

    return all_metrics


# ============================================================
# Visualization
# ============================================================

def plot_results(results, all_metrics):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    colors = {
        'SGD':         '#6b7280',
        'Adam':        '#3b82f6',
        'AdamW':       '#f59e0b',
        'Lion':        '#14b8a6',
        'AdaDelta':    '#ef4444',
        'LW-AdaDelta': '#10b981',
    }
    activity_names = [ACTIVITY_LABELS[i+1] for i in range(6)]
    short_names    = ['WALK', 'WALK_UP', 'WALK_DN',
                      'SIT', 'STAND', 'LAY']

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle(
        'UCI HAR: LW-AdaDelta vs Baselines\n'
        'Multi-Scale TCN, 9-channel inertial signals, 128 timesteps',
        fontsize=14, fontweight='bold'
    )

    # ── Plot 1: Overall Macro F1 bar chart ───────────────────────
    ax = axes[0, 0]
    f1_means = [all_metrics[o]['f1_mean'] for o in optimizers]
    f1_stds  = [all_metrics[o]['f1_std']  for o in optimizers]
    bar_colors = [colors[o] for o in optimizers]

    bars = ax.bar(optimizers, f1_means, yerr=f1_stds,
                  color=bar_colors, alpha=0.85, edgecolor='black',
                  linewidth=1.2, capsize=5)

    # Bold border on LW-AdaDelta and AdaDelta
    for bar, opt in zip(bars, optimizers):
        if opt in ('LW-AdaDelta', 'AdaDelta'):
            bar.set_linewidth(2.5)

    for bar, m, s in zip(bars, f1_means, f1_stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                m + s + 0.2,
                f'{m:.1f}%', ha='center', va='bottom',
                fontsize=8, fontweight='bold')

    ax.set_ylabel('Macro F1 (%)', fontsize=11)
    ax.set_title('Overall Test Macro F1', fontsize=12, fontweight='bold')
    ax.set_ylim([max(0, min(f1_means) - 5), min(100, max(f1_means) + 5)])
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)

    # ── Plot 2: Per-seed F1 improvement (LW vs AdaDelta) ─────────
    ax = axes[0, 1]
    lw_f1s  = [r['test_f1'] for r in results['LW-AdaDelta']]
    ada_f1s = [r['test_f1'] for r in results['AdaDelta']]
    diffs   = [lw - ada for lw, ada in zip(lw_f1s, ada_f1s)]
    labels  = [f'Seed {i}' for i in range(len(diffs))]

    bar_c = ['#10b981' if d > 0 else '#ef4444' for d in diffs]
    b     = ax.bar(labels, diffs, color=bar_c, alpha=0.85,
                   edgecolor='black', linewidth=1.5)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)

    for bar, d in zip(b, diffs):
        va  = 'bottom' if d >= 0 else 'top'
        off = max(abs(d) * 0.05, 0.05)
        off = off if d >= 0 else -off
        ax.text(bar.get_x() + bar.get_width()/2, d + off,
                f'{d:+.2f}%', ha='center', va=va,
                fontsize=11, fontweight='bold')

    mean_d = np.mean(diffs)
    ax.axhline(mean_d, color='purple', linestyle=':', linewidth=2,
               label=f'Mean Δ = {mean_d:+.2f}%')
    ax.legend(fontsize=10)
    ax.set_ylabel('ΔMacro F1 (LW-AdaDelta − AdaDelta)', fontsize=11)
    ax.set_title('Per-Seed Improvement over AdaDelta',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # ── Plot 3: Per-class F1 (LW-AdaDelta vs AdaDelta) ───────────
    ax = axes[0, 2]
    lw_cls  = all_metrics['LW-AdaDelta']['per_cls_f1']
    ada_cls = all_metrics['AdaDelta']['per_cls_f1']
    x       = np.arange(6)
    w       = 0.35

    ax.bar(x - w/2, ada_cls, w, label='AdaDelta',
           color=colors['AdaDelta'], alpha=0.8, edgecolor='black')
    ax.bar(x + w/2, lw_cls,  w, label='LW-AdaDelta',
           color=colors['LW-AdaDelta'], alpha=0.8, edgecolor='black')

    # Annotate delta
    for i in range(6):
        d = lw_cls[i] - ada_cls[i]
        y_pos = max(lw_cls[i], ada_cls[i]) + 0.5
        color = '#10b981' if d > 0 else '#ef4444'
        ax.text(i, y_pos, f'{d:+.1f}%', ha='center', va='bottom',
                fontsize=8, color=color, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('F1 Score (%)', fontsize=11)
    ax.set_title('Per-Class F1: LW-AdaDelta vs AdaDelta',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 115])

    # ── Plot 4: Val F1 training curve (median seed) ──────────────
    ax = axes[1, 0]
    med = len(results['Adam']) // 2
    for opt in optimizers:
        curve = results[opt][med]['history']['val_f1']
        lw = 2.5 if opt in ('LW-AdaDelta', 'AdaDelta') else 1.2
        al = 1.0 if opt in ('LW-AdaDelta', 'AdaDelta') else 0.55
        ax.plot(curve, label=opt, color=colors[opt],
                linewidth=lw, alpha=al)

    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Validation Macro F1 (%)', fontsize=11)
    ax.set_title('Validation F1 Curve (Median Seed)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Plot 5: Confusion matrix (LW-AdaDelta, median seed) ──────
    ax = axes[1, 1]
    med_run  = results['LW-AdaDelta'][len(results['LW-AdaDelta']) // 2]
    cm       = confusion_matrix(
        med_run['test_targets'], med_run['test_preds']
    )
    cm_norm  = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=short_names, yticklabels=short_names,
                ax=ax, linewidths=0.5, vmin=0, vmax=1,
                annot_kws={'size': 8})
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    ax.set_title('Confusion Matrix — LW-AdaDelta\n(Median Seed)',
                 fontsize=12, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

    # ── Plot 6: Convergence speed bar chart ──────────────────────
    ax = axes[1, 2]
    ep_means = [all_metrics[o]['ep_mean'] for o in optimizers]
    bars = ax.barh(optimizers, ep_means,
                   color=bar_colors, alpha=0.85,
                   edgecolor='black', linewidth=1.2)

    for bar, ep, opt in zip(bars, ep_means, optimizers):
        if opt in ('LW-AdaDelta', 'AdaDelta'):
            bar.set_linewidth(2.5)
        ax.text(ep + 0.5, bar.get_y() + bar.get_height()/2,
                f'{ep:.0f}', va='center', fontsize=9)

    ax.set_xlabel('Epochs to Convergence (Early Stop)', fontsize=11)
    ax.set_title('Convergence Speed', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    out_path = 'uci_har_results.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ Plot saved as '{out_path}'")
    plt.close()


# ============================================================
# Paper Summary Table
# ============================================================

def print_summary_table(all_metrics):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    activity_names = [ACTIVITY_LABELS[i+1] for i in range(6)]

    print("\n" + "="*80)
    print("PAPER-READY SUMMARY TABLE")
    print("="*80)

    # Overall metrics
    print(f"\n{'Optimizer':<15} {'Accuracy':<22} {'Macro F1':<22} {'Epochs'}")
    print("-"*70)
    best_f1 = max(all_metrics[o]['f1_mean'] for o in optimizers)
    for opt in optimizers:
        m   = all_metrics[opt]
        acc = f"{m['acc_mean']:.2f}±{m['acc_std']:.2f}%"
        f1  = f"{m['f1_mean']:.2f}±{m['f1_std']:.2f}%"
        if abs(m['f1_mean'] - best_f1) < 1e-6:
            f1 = f"*{f1}*"
        print(f"{opt:<15} {acc:<22} {f1:<22} {m['ep_mean']:.0f}")

    # Per-class F1 table
    print(f"\nPer-class F1 (mean across seeds):")
    header = f"{'Activity':<25} " + \
             "  ".join(f"{o:<14}" for o in optimizers)
    print(header)
    print("-" * len(header))
    for i, act in enumerate(activity_names):
        row = f"{act:<25} "
        best_cls = max(all_metrics[o]['per_cls_f1'][i] for o in optimizers)
        for o in optimizers:
            val = f"{all_metrics[o]['per_cls_f1'][i]:.1f}%"
            if abs(all_metrics[o]['per_cls_f1'][i] - best_cls) < 0.05:
                val = f"*{val}*"
            row += f"{val:<16}"
        print(row)

    print("\n* = best result per column/row")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # ── Step 1: Download & load ──────────────────────────────────
    DATA_DIR    = './uci_har_data'
    extract_dir = download_uci_har(DATA_DIR)
    X_train, y_train, X_test, y_test = load_uci_har(extract_dir)

    # ── Step 2: Benchmark ────────────────────────────────────────
    print("\n" + "="*80)
    print("UCI HAR Benchmark — Variable-Length Temporal Classification")
    print("="*80)
    print("Input    : 9 inertial channels × 128 timesteps (raw signals)")
    print("Model    : Multi-Scale TCN (kernels 3, 7, 15 + attention pool)")
    print("Classes  : 6 activities (3 dynamic + 3 static)")
    print("Metric   : Macro F1 (balanced across all classes)")
    print("Seeds    : 3")
    print("Early stopping: patience=15 on val Macro F1")
    print("="*80)

    results = run_full_benchmark(
        X_train, y_train, X_test, y_test,
        device=device, seeds=[0, 1, 2]
    )

    # ── Step 3: Analyze ──────────────────────────────────────────
    all_metrics = analyze_results(results)

    # ── Step 4: Plot ─────────────────────────────────────────────
    plot_results(results, all_metrics)

    # ── Step 5: Paper table ──────────────────────────────────────
    print_summary_table(all_metrics)

    print("\n" + "="*80)
    print("BENCHMARK COMPLETE!")
    print("="*80)
    print("\nKey claims to verify:")
    print("  • LW-AdaDelta F1 > AdaDelta F1 overall")
    print("  • Largest per-class wins expected on transition activities:")
    print("    WALKING_UPSTAIRS and WALKING_DOWNSTAIRS (hardest classes)")
    print("  • Confusion matrix: check SITTING/STANDING confusion —")
    print("    LW-AdaDelta should confuse these less (static activity boundary)")
    print("\n✓ Results ready for paper!")

Using device: cuda
[INFO] Downloading UCI HAR Dataset (~60 MB) …
[INFO] Downloaded → ./uci_har_data/uci_har.zip
[INFO] Extracting …
[INFO] Extracted → ./uci_har_data/UCI HAR Dataset
[INFO] train: X=(7352, 9, 128), y=(7352,), classes=[0, 1, 2, 3, 4, 5]
[INFO] test: X=(2947, 9, 128), y=(2947,), classes=[0, 1, 2, 3, 4, 5]

[INFO] After normalization:
       Train: (7352, 9, 128)  Test: (2947, 9, 128)
       Class distribution (train): {'WALKING': 1226, 'WALKING_UPSTAIRS': 1073, 'WALKING_DOWNSTAIRS': 986, 'SITTING': 1286, 'STANDING': 1374, 'LAYING': 1407}

UCI HAR Benchmark — Variable-Length Temporal Classification
Input    : 9 inertial channels × 128 timesteps (raw signals)
Model    : Multi-Scale TCN (kernels 3, 7, 15 + attention pool)
Classes  : 6 activities (3 dynamic + 3 static)
Metric   : Macro F1 (balanced across all classes)
Seeds    : 3
Early stopping: patience=15 on val Macro F1

[Progress 1/18] SGD  seed=0

Optimizer=SGD  Seed=0
[INFO] Loaders — Train: 6249, Val: 1103, Test: 2947